# VIX Replication Using ThetaData

Version: 0.1

Goal:
Replicate standard 30-day VIX using historical SPX/SPXW option chains from ThetaData.

Status:
- One-date replication works.
- Tested on 2024-01-16.
- Replicated VIX: 13.7747.
- Official Cboe VIX close: 13.84.
- Difference: -0.0653 VIX points.

Runtime:
- One date takes about 2 minutes.
- This is acceptable for validation.
- Before running full history, we need caching / batching.

In [1]:
import requests
import pandas as pd
import numpy as np

from datetime import datetime
from math import exp, sqrt

BASE_URL = "http://127.0.0.1:25510/v2"

CALC_TIME_MS = 57600000  # 16:00:00 ET
DEFAULT_RISK_FREE_RATE = 0.05

In [2]:
def td_get(endpoint, params, timeout=180):
    """
    Helper function to call ThetaData's local REST API.
    Theta Terminal must be running and logged in.
    """
    url = BASE_URL + endpoint

    r = requests.get(url, params=params, timeout=timeout)
    r.raise_for_status()

    data = r.json()

    if data.get("header", {}).get("error_type") is not None:
        raise RuntimeError(data["header"])

    return data

In [3]:
def yyyymmdd_to_date(x):
    return datetime.strptime(str(int(x)), "%Y%m%d").date()

In [4]:
def list_expirations(root):
    data = td_get("/list/expirations", {"root": root})
    exps = data["response"]
    return sorted([int(x) for x in exps])


spx_exps = list_expirations("SPX")
spxw_exps = list_expirations("SPXW")

print("SPX expirations loaded:", len(spx_exps))
print("SPXW expirations loaded:", len(spxw_exps))

SPX expirations loaded: 195
SPXW expirations loaded: 2184


In [5]:
# ============================================================
# Core VIX replication functions
# Paste this entire cell AFTER the list_expirations cell
# ============================================================

CALC_TIME_MS = 57600000          # 16:00:00 ET
DEFAULT_RISK_FREE_RATE = 0.05    # temporary simplification


def choose_vix_expirations(trade_date_yyyymmdd):
    trade_dt = yyyymmdd_to_date(trade_date_yyyymmdd)

    candidates = []

    # Standard SPX expirations
    for expi in spx_exps:
        exp_dt = yyyymmdd_to_date(expi)
        if exp_dt > trade_dt:
            candidates.append(("SPX", expi, (exp_dt - trade_dt).days))

    # SPXW Friday expirations
    for expi in spxw_exps:
        exp_dt = yyyymmdd_to_date(expi)
        if exp_dt > trade_dt and exp_dt.weekday() == 4:
            candidates.append(("SPXW", expi, (exp_dt - trade_dt).days))

    candidates = pd.DataFrame(candidates, columns=["root", "exp", "days"])
    candidates = candidates.sort_values("days")

    near = candidates[candidates["days"] <= 30].tail(1)
    next_ = candidates[candidates["days"] > 30].head(1)

    if near.empty or next_.empty:
        raise ValueError("Could not find near and next expirations around 30 days.")

    return pd.concat([near, next_]).reset_index(drop=True)


QUOTE_COLUMNS = [
    "ms_of_day", "bid_size", "bid_exchange", "bid", "bid_condition",
    "ask_size", "ask_exchange", "ask", "ask_condition", "date"
]


def get_chain_at_time(root, expi, trade_date, calc_time_ms):
    data = td_get(
        "/bulk_at_time/option/quote",
        {
            "root": root,
            "exp": expi,
            "start_date": trade_date,
            "end_date": trade_date,
            "ivl": calc_time_ms
        }
    )

    rows = []

    for item in data["response"]:
        contract = item["contract"]
        ticks = item["ticks"]

        if len(ticks) == 0:
            continue

        tick = ticks[-1]
        row = dict(zip(QUOTE_COLUMNS, tick))

        row["root"] = contract["root"]
        row["expiration"] = contract["expiration"]
        row["strike"] = contract["strike"] / 1000
        row["right"] = contract["right"]

        rows.append(row)

    df = pd.DataFrame(rows)

    if df.empty:
        raise ValueError(f"No data returned for {root} {expi} on {trade_date}")

    df["mid"] = (df["bid"] + df["ask"]) / 2

    return df


def minutes_to_expiry_approx(trade_date_yyyymmdd, exp_yyyymmdd, calc_time_ms):
    trade_dt = yyyymmdd_to_date(trade_date_yyyymmdd)
    exp_dt = yyyymmdd_to_date(exp_yyyymmdd)

    calc_minutes = calc_time_ms / 1000 / 60
    expiry_minutes = 16 * 60  # 4:00 PM ET approximation

    calendar_days = (exp_dt - trade_dt).days

    return int(calendar_days * 1440 + expiry_minutes - calc_minutes)


def calc_single_term_variance(chain, minutes_to_expiry, r=0.05):
    T = minutes_to_expiry / 525600  # 365 * 1440 minutes

    df = chain.copy()

    # Basic quote cleaning
    df = df[(df["bid"] >= 0) & (df["ask"] > 0) & (df["ask"] >= df["bid"])]
    df = df[["strike", "right", "bid", "ask", "mid"]]

    calls = df[df["right"] == "C"].set_index("strike")
    puts = df[df["right"] == "P"].set_index("strike")

    common_strikes = calls.index.intersection(puts.index)

    cp = pd.DataFrame({
        "call_mid": calls.loc[common_strikes, "mid"],
        "put_mid": puts.loc[common_strikes, "mid"],
        "call_bid": calls.loc[common_strikes, "bid"],
        "put_bid": puts.loc[common_strikes, "bid"],
        "call_ask": calls.loc[common_strikes, "ask"],
        "put_ask": puts.loc[common_strikes, "ask"],
    }).sort_index()

    cp["abs_diff"] = (cp["call_mid"] - cp["put_mid"]).abs()
    atm_strike = cp["abs_diff"].idxmin()

    # Forward from put-call parity
    F = atm_strike + exp(r * T) * (
        cp.loc[atm_strike, "call_mid"] - cp.loc[atm_strike, "put_mid"]
    )

    # K0 = strike immediately below forward
    strikes = np.array(sorted(common_strikes))
    k0_candidates = strikes[strikes <= F]

    if len(k0_candidates) == 0:
        raise ValueError("No K0 found below forward.")

    K0 = k0_candidates[-1]

    selected_rows = []

    # OTM puts below K0
    put_strikes = sorted([k for k in puts.index if k < K0], reverse=True)
    zero_count = 0

    for k in put_strikes:
        bid = puts.loc[k, "bid"]
        ask = puts.loc[k, "ask"]

        if bid <= 0 or ask <= 0:
            zero_count += 1
            if zero_count >= 2:
                break
            continue

        zero_count = 0
        selected_rows.append({"strike": k, "Q": puts.loc[k, "mid"]})

    # OTM calls above K0
    call_strikes = sorted([k for k in calls.index if k > K0])
    zero_count = 0

    for k in call_strikes:
        bid = calls.loc[k, "bid"]
        ask = calls.loc[k, "ask"]

        if bid <= 0 or ask <= 0:
            zero_count += 1
            if zero_count >= 2:
                break
            continue

        zero_count = 0
        selected_rows.append({"strike": k, "Q": calls.loc[k, "mid"]})

    # At K0, use average of call and put midpoint
    k0_q = 0.5 * (calls.loc[K0, "mid"] + puts.loc[K0, "mid"])
    selected_rows.append({"strike": K0, "Q": k0_q})

    selected = (
        pd.DataFrame(selected_rows)
        .drop_duplicates("strike")
        .sort_values("strike")
        .reset_index(drop=True)
    )

    selected["delta_k"] = np.nan

    for i in range(len(selected)):
        if i == 0:
            selected.loc[i, "delta_k"] = (
                selected.loc[i + 1, "strike"] - selected.loc[i, "strike"]
            )
        elif i == len(selected) - 1:
            selected.loc[i, "delta_k"] = (
                selected.loc[i, "strike"] - selected.loc[i - 1, "strike"]
            )
        else:
            selected.loc[i, "delta_k"] = (
                selected.loc[i + 1, "strike"] - selected.loc[i - 1, "strike"]
            ) / 2

    selected["contribution"] = (
        selected["delta_k"]
        / (selected["strike"] ** 2)
        * exp(r * T)
        * selected["Q"]
    )

    variance = (
        (2 / T) * selected["contribution"].sum()
        - (1 / T) * ((F / K0) - 1) ** 2
    )

    return {
        "variance": variance,
        "vol": sqrt(max(variance, 0)) * 100,
        "T": T,
        "minutes": minutes_to_expiry,
        "F": F,
        "K0": K0,
        "selected_options": selected
    }


def interpolate_to_30d(term_df):
    near = term_df.iloc[0]
    next_ = term_df.iloc[1]

    M1 = near["minutes"]
    M2 = next_["minutes"]

    M30 = 30 * 1440
    M365 = 365 * 1440

    T1 = M1 / M365
    T2 = M2 / M365

    var1 = near["variance"]
    var2 = next_["variance"]

    variance_30d = (
        T1 * var1 * ((M2 - M30) / (M2 - M1))
        + T2 * var2 * ((M30 - M1) / (M2 - M1))
    ) * (M365 / M30)

    vix = 100 * sqrt(max(variance_30d, 0))

    return vix


def calculate_vix_for_date(trade_date, calc_time_ms=CALC_TIME_MS, r=DEFAULT_RISK_FREE_RATE):
    terms = choose_vix_expirations(trade_date)

    chains = []

    for _, row in terms.iterrows():
        print("Pulling:", row["root"], int(row["exp"]))

        chain = get_chain_at_time(
            root=row["root"],
            expi=int(row["exp"]),
            trade_date=trade_date,
            calc_time_ms=calc_time_ms
        )

        chain["days"] = row["days"]
        chains.append(chain)

    raw_options = pd.concat(chains, ignore_index=True)

    term_results = []

    for _, row in terms.iterrows():
        root = row["root"]
        expi = int(row["exp"])

        chain = raw_options[
            (raw_options["root"] == root) &
            (raw_options["expiration"] == expi)
        ]

        mins = minutes_to_expiry_approx(trade_date, expi, calc_time_ms)

        result = calc_single_term_variance(chain, mins, r=r)

        term_results.append({
            "root": root,
            "exp": expi,
            "days": row["days"],
            "minutes": mins,
            "variance": result["variance"],
            "vol": result["vol"],
            "F": result["F"],
            "K0": result["K0"],
            "num_selected_options": len(result["selected_options"])
        })

    term_df = pd.DataFrame(term_results)

    vix_value = interpolate_to_30d(term_df)

    return {
        "trade_date": trade_date,
        "calc_time_ms": calc_time_ms,
        "vix": vix_value,
        "term_df": term_df,
        "raw_options": raw_options
    }


def load_cboe_vix_history():
    cboe_url = "https://cdn.cboe.com/api/global/us_indices/daily_prices/VIX_History.csv"
    cboe_vix = pd.read_csv(cboe_url)
    cboe_vix["DATE"] = pd.to_datetime(cboe_vix["DATE"])
    cboe_vix["date_int"] = cboe_vix["DATE"].dt.strftime("%Y%m%d").astype(int)
    return cboe_vix


def compare_to_cboe_close(trade_date, replicated_vix):
    cboe_vix = load_cboe_vix_history()

    row = cboe_vix[cboe_vix["date_int"] == int(trade_date)]

    if row.empty:
        raise ValueError(f"No Cboe VIX close found for {trade_date}")

    official_vix_close = float(row["CLOSE"].iloc[0])
    diff = replicated_vix - official_vix_close

    return {
        "date": trade_date,
        "replicated_vix": replicated_vix,
        "official_vix_close": official_vix_close,
        "difference": diff,
        "abs_difference": abs(diff)
    }

In [6]:
result = calculate_vix_for_date(20240116)

print("Calculated VIX:", round(result["vix"], 4))
display(result["term_df"])

comparison = compare_to_cboe_close(20240116, result["vix"])
comparison

Pulling: SPXW 20240209
Pulling: SPX 20240216
Calculated VIX: 13.7747


,root,exp,days,minutes,variance,vol,F,K0,num_selected_options
0,SPXW,20240209,24,34560,0.018789,13.707176,4781.504940,4780.0,161
1,SPX,20240216,31,44640,0.018998,13.783338,4783.694468,4780.0,351


{'date': 20240116,
 'replicated_vix': 13.774654750579229,
 'official_vix_close': 13.84,
 'difference': -0.06534524942077091,
 'abs_difference': 0.06534524942077091}